### Sales&Profit Performance MV

In [0]:
CREATE OR REFRESH MATERIALIZED VIEW mavenmarket.gold.gold_sales_profit_performance
COMMENT "Materialized View: Gold layer aggregated monthly sales performance with time intelligence metrics"
AS
WITH base_monthly_sales AS (
    SELECT 
        -- Monthly Date Groupings (from Gold dim_date)
        d.year_number,
        d.quarter_number,
        d.month_number,
        d.month_name,
        d.year_month_code,

        -- Store Attributes (from Gold dim_store)
        s.store_sk,
        s.store_id,
        s.store_name,
        s.store_type_description,
        s.city_name AS store_city,
        s.state_province_code AS store_state,
        s.country_name AS store_country,

        -- Region Attributes (from Gold dim_region joined via region_sk)
        r.region_sk,
        r.region_id,
        r.sales_region_name,
        r.sales_district_name,

        -- Product Attributes (from Gold dim_product)
        p.product_sk,
        p.product_id,
        p.product_name,
        p.brand_name,
        p.category_name,
        p.sku_code,

        -- Transaction Channel (from Gold fact_sales)
        f.transaction_mode,

        -- Monthly Aggregated Financial Measures (pre-calculated in Gold fact_sales)
        COUNT(DISTINCT f.customer_sk) AS total_orders_count,
        SUM(f.sales_quantity) AS total_units_sold,
        SUM(f.gross_revenue_amount) AS gross_revenue_amount,
        SUM(f.total_cost_amount) AS total_cost_amount,
        SUM(f.gross_profit_amount) AS gross_profit_amount

    FROM LIVE.fact_sales f
    
    -- Join Gold Date Dimension using integer surrogate key
    JOIN LIVE.dim_date d 
        ON f.transaction_date_sk = d.date_sk
        
    -- Join Gold Store Dimension using string surrogate key
    JOIN LIVE.dim_store s 
        ON f.store_sk = s.store_sk
        
    -- Join Gold Region Dimension using string surrogate key
    JOIN LIVE.dim_region r 
        ON s.region_sk = r.region_sk
        
    -- Join Gold Product Dimension using string surrogate key
    JOIN LIVE.dim_product p 
        ON f.product_sk = p.product_sk

    GROUP BY ALL
),
sales_with_time_intelligence AS (
    SELECT 
        *,
        -- Profit Margin Percentage
        ROUND(
            (gross_profit_amount / NULLIF(gross_revenue_amount, 0)) * 100.0, 
            2
        ) AS profit_margin_pct,

        -- Average Order Value (AOV)
        ROUND(
            gross_revenue_amount / NULLIF(total_orders_count, 0), 
            2
        ) AS average_order_value,

        -- Prior Year Monthly Revenue (Looks back 12 true monthly partition buckets)
        LAG(gross_revenue_amount, 12) OVER (
            PARTITION BY store_id, product_id, transaction_mode 
            ORDER BY year_number, month_number
        ) AS prior_year_gross_revenue,

        -- Prior Year Monthly Profit
        LAG(gross_profit_amount, 12) OVER (
            PARTITION BY store_id, product_id, transaction_mode 
            ORDER BY year_number, month_number
        ) AS prior_year_gross_profit

    FROM base_monthly_sales
)
SELECT 
    *,
    -- Revenue YoY Growth Calculations
    CASE 
        WHEN prior_year_gross_revenue IS NULL THEN NULL
        ELSE ROUND(gross_revenue_amount - prior_year_gross_revenue, 2)
    END AS revenue_yoy_growth_amount,
    
    ROUND(
        ((gross_revenue_amount - prior_year_gross_revenue) / NULLIF(prior_year_gross_revenue, 0)) * 100.0, 
        2
    ) AS revenue_yoy_growth_pct,

    -- Profit YoY Growth Calculations
    CASE 
        WHEN prior_year_gross_profit IS NULL THEN NULL
        ELSE ROUND(gross_profit_amount - prior_year_gross_profit, 2)
    END AS profit_yoy_growth_amount,

    ROUND(
        ((gross_profit_amount - prior_year_gross_profit) / NULLIF(prior_year_gross_profit, 0)) * 100.0, 
        2
    ) AS profit_yoy_growth_pct

FROM sales_with_time_intelligence;

### Customer Analytics Performance MV

In [0]:
CREATE OR REFRESH MATERIALIZED VIEW mavenmarket.gold.gold_customer_analytics_performance
COMMENT "Materialized View: Gold layer aggregated customer performance dynamic across all dim_customer entries"
AS
WITH master_customers AS (
    -- Guarantees complete customer coverage directly from Gold dim_customer
    SELECT 
        customer_sk,
        customer_id,
        full_name AS customer_name,
        city_name AS customer_city,
        state_province_code AS customer_state,
        country_name AS customer_country,
        gender_code AS gender,
        marital_status_code AS marital_status,
        yearly_income_range AS yearly_income_amount,
        education_level,
        occupation_name,
        homeowner_flag AS is_homeowner_flag
    FROM LIVE.dim_customer
    WHERE customer_id IS NOT NULL
      AND is_current_flag = TRUE
),
customer_store_preference AS (
    -- Calculate primary region and store for active customers using Gold fact_sales
    SELECT 
        f.customer_sk,
        s.store_sk,
        r.region_sk,
        r.region_id,
        r.sales_region_name,
        r.sales_district_name,
        ROW_NUMBER() OVER (
            PARTITION BY f.customer_sk 
            ORDER BY COUNT(DISTINCT CONCAT(f.customer_sk, '_', f.transaction_date_sk, '_', f.store_sk)) DESC, 
                     MAX(f.transaction_date_sk) DESC, 
                     s.store_sk ASC
        ) AS preference_rank
    FROM LIVE.fact_sales f
    LEFT JOIN LIVE.dim_store s 
        ON f.store_sk = s.store_sk
    LEFT JOIN LIVE.dim_region r 
        ON s.region_sk = r.region_sk
    GROUP BY 
        f.customer_sk,
        s.store_sk,
        r.region_sk,
        r.region_id,
        r.sales_region_name,
        r.sales_district_name
),
customer_primary_region AS (
    SELECT 
        customer_sk,
        region_id,
        sales_region_name,
        sales_district_name
    FROM customer_store_preference
    WHERE preference_rank = 1
),
customer_sales_summary AS (
    -- Aggregate purchases per customer from Gold fact_sales and dim_date
    SELECT 
        f.customer_sk,
        MIN(d.calendar_date) AS first_purchase_date,
        MAX(d.calendar_date) AS most_recent_purchase_date,
        COUNT(DISTINCT CONCAT(f.customer_sk, '_', f.transaction_date_sk, '_', f.store_sk)) AS total_orders_count,
        COUNT(DISTINCT f.transaction_date_sk) AS active_purchase_days_count,
        COUNT(DISTINCT f.product_sk) AS distinct_products_bought,
        SUM(f.sales_quantity) AS total_units_purchased,
        SUM(f.gross_revenue_amount) AS customer_lifetime_value,
        SUM(f.total_cost_amount) AS total_customer_cost,
        SUM(f.gross_profit_amount) AS total_profit_generated
    FROM LIVE.fact_sales f
    LEFT JOIN LIVE.dim_date d 
        ON f.transaction_date_sk = d.date_sk
    GROUP BY f.customer_sk
),
customer_sales_base AS (
    SELECT 
        -- Complete Master Customer Attributes
        c.customer_sk,
        c.customer_id,
        c.customer_name,
        c.customer_city,
        c.customer_state,
        c.customer_country,
        c.gender,
        c.marital_status,
        c.yearly_income_amount,
        c.education_level,
        c.occupation_name,
        c.is_homeowner_flag,

        -- Primary Region Details
        pr.region_id,
        pr.sales_region_name,
        pr.sales_district_name,

        -- Purchase Timelines
        s.first_purchase_date,
        s.most_recent_purchase_date,
        
        -- Aggregated Order Metrics
        COALESCE(s.total_orders_count, 0) AS total_orders_count,
        COALESCE(s.active_purchase_days_count, 0) AS active_purchase_days_count,
        COALESCE(s.distinct_products_bought, 0) AS distinct_products_bought,
        COALESCE(s.total_units_purchased, 0) AS total_units_purchased,

        -- Pre-calculated Financial Measures from Gold Fact
        COALESCE(s.customer_lifetime_value, 0.00) AS customer_lifetime_value,
        COALESCE(s.total_customer_cost, 0.00) AS total_customer_cost,
        COALESCE(s.total_profit_generated, 0.00) AS total_profit_generated

    FROM master_customers c
    LEFT JOIN customer_sales_summary s 
        ON c.customer_sk = s.customer_sk
    LEFT JOIN customer_primary_region pr 
        ON c.customer_sk = pr.customer_sk
),
customer_analytics_calculated AS (
    SELECT 
        b.*,

        -- Portfolio Counts
        COUNT(b.customer_sk) OVER () AS total_customer_count,
        COUNT(CASE WHEN b.total_orders_count > 0 THEN b.customer_sk END) OVER () AS total_active_customers_count,

        -- Financial & Order Ratios
        ROUND(
            b.customer_lifetime_value / NULLIF(b.total_orders_count, 0), 
            2
        ) AS avg_order_value,

        ROUND(
            (b.total_profit_generated / NULLIF(b.customer_lifetime_value, 0)) * 100.0, 
            2
        ) AS customer_profit_margin_pct,

        ROUND(
            b.total_units_purchased / NULLIF(b.total_orders_count, 0), 
            2
        ) AS avg_units_per_order,

        DATEDIFF(CURRENT_DATE(), b.most_recent_purchase_date) AS days_since_last_purchase,

        -- Order Frequency Tiers
        CASE 
            WHEN b.total_orders_count >= 20 THEN 'High Frequency (20+ Orders)'
            WHEN b.total_orders_count BETWEEN 8 AND 19 THEN 'Medium Frequency (8-19 Orders)'
            WHEN b.total_orders_count BETWEEN 1 AND 7 THEN 'Low Frequency (1-7 Orders)'
            ELSE 'No Purchases Yet'
        END AS purchase_frequency_tier,

        -- Portfolio Averages
        ROUND(
            AVG(b.customer_lifetime_value) OVER (), 
            2
        ) AS global_avg_customer_ltv

    FROM customer_sales_base b
)
SELECT 
    c.*,
    DENSE_RANK() OVER (ORDER BY c.customer_lifetime_value DESC) AS ltv_rank_position,
    ROUND(c.customer_lifetime_value - c.global_avg_customer_ltv, 2) AS ltv_variance_from_avg
FROM customer_analytics_calculated c;

### Returns Performance MV

In [0]:
CREATE OR REFRESH MATERIALIZED VIEW mavenmarket.gold.gold_returns_performance
COMMENT "Materialized View: Gold layer aggregated return performance with monthly and all-time sales fallback metrics"
AS
WITH returns_base AS (
    -- Aggregate returns directly from Gold fact_returns using surrogate keys
    SELECT 
        fr.return_date_key,
        fr.store_sk,
        fr.product_sk,
        SUM(fr.return_quantity) AS returned_quantity
    FROM LIVE.fact_returns fr
    WHERE fr.return_quantity > 0
    GROUP BY 
        fr.return_date_key,
        fr.store_sk,
        fr.product_sk
),
monthly_product_sales AS (
    -- Aggregate sales at Year + Month level for Return Rate % denominator using Gold fact_sales
    SELECT 
        d.year_number,
        d.month_number,
        fs.store_sk,
        fs.product_sk,
        SUM(fs.sales_quantity) AS monthly_sales_quantity,
        SUM(fs.gross_revenue_amount) AS monthly_sales_revenue
    FROM LIVE.fact_sales fs
    JOIN LIVE.dim_date d 
        ON fs.transaction_date_sk = d.date_sk
    GROUP BY 
        d.year_number,
        d.month_number,
        fs.store_sk,
        fs.product_sk
),
overall_product_sales AS (
    -- All-time product sales fallback for items returned outside their purchase month
    SELECT 
        fs.product_sk,
        SUM(fs.sales_quantity) AS total_product_sales_quantity,
        SUM(fs.gross_revenue_amount) AS total_product_sales_revenue
    FROM LIVE.fact_sales fs
    GROUP BY fs.product_sk
)
SELECT 
    -- 1. Date & Trend Attributes (from Gold dim_date)
    d.calendar_date AS return_date,
    d.year_number AS return_year,
    d.quarter_number AS return_quarter,
    d.month_number AS return_month_number,
    d.month_name AS return_month_name,
    d.year_month_code AS return_year_month,
    d.day_name AS return_day_of_week,

    -- 2. Product & Category Attributes (from Gold dim_product)
    p.product_sk,
    p.product_id,
    p.product_name,
    p.brand_name,
    p.category_name AS product_category,
    p.sku_code,
    p.retail_price_amount AS unit_retail_price,
    p.unit_cost_amount AS unit_cost,

    -- 3. Store & Region Attributes (from Gold dim_store and dim_region)
    st.store_sk,
    st.store_id,
    st.store_name,
    st.city_name AS store_city,
    st.state_province_code AS store_state,
    r.region_sk,
    r.sales_region_name AS sales_region,
    r.sales_district_name AS sales_district,

    -- 4. Volumetric Measures
    rb.returned_quantity,
    COALESCE(ms.monthly_sales_quantity, 0) AS monthly_gross_sales_quantity,
    COALESCE(ops.total_product_sales_quantity, 0) AS total_product_sales_quantity,

    -- 5. Financial Return Value Calculations (derived from Gold dim_product pricing)
    CAST(rb.returned_quantity * COALESCE(p.retail_price_amount, 0.00) AS DECIMAL(18,2)) AS total_return_value_amount,
    CAST(rb.returned_quantity * COALESCE(p.unit_cost_amount, 0.00) AS DECIMAL(18,2)) AS total_return_cost_amount,

    -- 6. Fully Null-Safe Key Ratios & KPI Calculations
    -- Return Rate % (Uses Monthly Store Sales -> Fallback to Product Total Sales -> Fallback to 100.00%)
    CASE 
        WHEN COALESCE(ms.monthly_sales_quantity, 0) > 0 THEN 
            ROUND((rb.returned_quantity * 100.0) / ms.monthly_sales_quantity, 2)
        WHEN COALESCE(ops.total_product_sales_quantity, 0) > 0 THEN 
            ROUND((rb.returned_quantity * 100.0) / ops.total_product_sales_quantity, 2)
        ELSE 100.00
    END AS return_rate_pct,

    -- Return Value % (Uses Monthly Store Revenue -> Fallback to Product Total Revenue -> Fallback to 100.00%)
    CASE 
        WHEN COALESCE(ms.monthly_sales_revenue, 0) > 0 THEN 
            ROUND(((rb.returned_quantity * COALESCE(p.retail_price_amount, 0.00)) * 100.0) / ms.monthly_sales_revenue, 2)
        WHEN COALESCE(ops.total_product_sales_revenue, 0) > 0 THEN 
            ROUND(((rb.returned_quantity * COALESCE(p.retail_price_amount, 0.00)) * 100.0) / ops.total_product_sales_revenue, 2)
        ELSE 100.00
    END AS return_value_pct

FROM returns_base rb

-- Primary Date Join via Integer Surrogate Key
JOIN LIVE.dim_date d 
    ON rb.return_date_key = d.date_sk

-- Product & Store Joins via String Surrogate Keys
JOIN LIVE.dim_product p 
    ON rb.product_sk = p.product_sk

JOIN LIVE.dim_store st 
    ON rb.store_sk = st.store_sk

-- Region Join via String Surrogate Key from dim_store
JOIN LIVE.dim_region r 
    ON st.region_sk = r.region_sk

-- Monthly Sales Join using Date attributes and Surrogate Keys
LEFT JOIN monthly_product_sales ms
    ON d.year_number = ms.year_number
   AND d.month_number = ms.month_number
   AND rb.store_sk = ms.store_sk
   AND rb.product_sk = ms.product_sk

-- Overall Product Sales Fallback Join via product_sk
LEFT JOIN overall_product_sales ops
    ON rb.product_sk = ops.product_sk;

### Online Orders Performance MV

In [0]:
CREATE OR REFRESH MATERIALIZED VIEW mavenmarket.gold.gold_online_orders_performance
COMMENT "Materialized View: Gold layer aggregated online sales channel metrics, fulfillment performance, customer basket behavior, and return analytics"
AS
WITH clean_online_transactions AS (
    -- Filter strictly for online transactions from Gold fact_sales
    SELECT 
        f.customer_sk,
        f.store_sk,
        f.product_sk,
        f.transaction_date_sk,
        f.sales_quantity AS quantity,
        f.transaction_mode,
        f.gross_revenue_amount,
        f.total_cost_amount,
        f.gross_profit_amount
    FROM LIVE.fact_sales f
    WHERE TRIM(f.transaction_mode) = 'online'
      AND f.transaction_date_sk IS NOT NULL
),
online_returns AS (
    -- Aggregate returned units from Gold fact_returns using surrogate keys
    SELECT 
        r.product_sk,
        r.store_sk,
        r.return_date_key,
        SUM(r.return_quantity) AS total_returned_units
    FROM LIVE.fact_returns r
    WHERE r.product_sk IS NOT NULL
    GROUP BY ALL
),
base_online_monthly_performance AS (
    SELECT 
        -- Monthly Calendar Attributes (from Gold dim_date)
        d.year_number,
        d.quarter_number,
        d.month_number,
        d.month_name,
        d.year_month_code,

        -- Store & Fulfillment Region (from Gold dim_store)
        s.store_sk,
        s.store_id,
        s.store_name,
        s.store_type_description,
        s.city_name AS store_city,
        s.state_province_code AS store_state,
        s.country_name AS store_country,

        -- Region Attributes (from Gold dim_region joined via region_sk)
        r.region_sk,
        r.region_id,
        r.sales_region_name,
        r.sales_district_name,

        -- Product Attributes (from Gold dim_product)
        p.product_sk,
        p.product_id,
        p.product_name,
        p.brand_name,
        p.category_name,
        p.sku_code,

        -- Channel Type (Fixed to 'online')
        t.transaction_mode AS channel_type,

        -- Online Aggregated Financial & Volume Metrics
        COUNT(DISTINCT t.customer_sk) AS distinct_online_buyers_count,
        COUNT(DISTINCT CONCAT(t.customer_sk, '_', t.transaction_date_sk, '_', t.store_sk)) AS total_online_orders_count,
        SUM(t.quantity) AS total_online_units_sold,
        COALESCE(SUM(ret.total_returned_units), 0) AS total_online_units_returned,

        -- Pre-calculated Financial Metrics from Gold Fact
        SUM(t.gross_revenue_amount) AS gross_online_revenue_amount,
        SUM(t.total_cost_amount) AS total_online_cost_amount,
        SUM(t.gross_profit_amount) AS gross_online_profit_amount

    FROM clean_online_transactions t
    JOIN LIVE.dim_date d 
        ON t.transaction_date_sk = d.date_sk
    LEFT JOIN LIVE.dim_store s 
        ON t.store_sk = s.store_sk
    LEFT JOIN LIVE.dim_region r 
        ON s.region_sk = r.region_sk
    LEFT JOIN LIVE.dim_product p 
        ON t.product_sk = p.product_sk
    LEFT JOIN online_returns ret 
        ON t.product_sk = ret.product_sk 
       AND t.store_sk = ret.store_sk 
       AND t.transaction_date_sk = ret.return_date_key

    GROUP BY ALL
),
online_with_calculated_kpis AS (
    SELECT 
        *,

        -- Online Financial Ratios
        ROUND(
            (gross_online_profit_amount / NULLIF(gross_online_revenue_amount, 0)) * 100.0, 
            2
        ) AS online_profit_margin_pct,

        -- Average Online Order Value (AOV)
        ROUND(
            gross_online_revenue_amount / NULLIF(total_online_orders_count, 0), 
            2
        ) AS online_average_order_value,

        -- Average Units per Online Basket
        ROUND(
            total_online_units_sold / NULLIF(total_online_orders_count, 0), 
            2
        ) AS avg_units_per_online_order,

        -- Online Return Rate Percentage
        ROUND(
            (total_online_units_returned / NULLIF(total_online_units_sold, 0)) * 100.0, 
            2
        ) AS online_return_rate_pct,

        -- Prior Year Online Revenue (12-month lookback using surrogate keys)
        LAG(gross_online_revenue_amount, 12) OVER (
            PARTITION BY store_sk, product_sk 
            ORDER BY year_number, month_number
        ) AS prior_year_online_revenue,

        -- Prior Year Online Profit
        LAG(gross_online_profit_amount, 12) OVER (
            PARTITION BY store_sk, product_sk 
            ORDER BY year_number, month_number
        ) AS prior_year_online_profit

    FROM base_online_monthly_performance
)
SELECT 
    *,
    -- Year-over-Year Online Growth Metrics
    CASE 
        WHEN prior_year_online_revenue IS NULL THEN NULL
        ELSE ROUND(gross_online_revenue_amount - prior_year_online_revenue, 2)
    END AS online_revenue_yoy_growth_amount,
    
    ROUND(
        ((gross_online_revenue_amount - prior_year_online_revenue) / NULLIF(prior_year_online_revenue, 0)) * 100.0, 
        2
    ) AS online_revenue_yoy_growth_pct,

    CASE 
        WHEN prior_year_online_profit IS NULL THEN NULL
        ELSE ROUND(gross_online_profit_amount - prior_year_online_profit, 2)
    END AS online_profit_yoy_growth_amount,

    ROUND(
        ((gross_online_profit_amount - prior_year_online_profit) / NULLIF(prior_year_online_profit, 0)) * 100.0, 
        2
    ) AS online_profit_yoy_growth_pct

FROM online_with_calculated_kpis;